# ML-08 — Model Training and Benchmark Evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This notebook trains supervised learning models (Logistic Regression, Decision Tree, Random Forest) using a client-holdout split strategy.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score

ROOT = Path('.').resolve()
while not (ROOT / 'data').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

# Load dataset
df = pd.read_csv(ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv')
df['target'] = (df['trend_direction'] == 'down').astype(int)

# Features
features = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'days_with_impressions',
    'days_since_last_update', 'content_age_days', 'ctr', 'avg_position',
    'engagement_rate', 'scroll_rate'
]

X = df[features].fillna(0)
y = df['target']
groups = df['client_id']

# Group Split by Client
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

# Train Models
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)

# Test Predictions & Precision@50
test_df = df.iloc[test_idx].copy()
test_df['rf_prob'] = rf.predict_proba(X_test)[:, 1]

top50_rf = test_df.sort_values('rf_prob', ascending=False).head(50)
rf_p50 = top50_rf['target'].mean()

print("=== MODEL BENCHMARK RESULTS ===")
print(f"Train Set Size: {len(X_train):,} rows | Test Set Size: {len(X_test):,} rows")
print(f"Random Forest Precision@50 (Client Holdout): {rf_p50:.3f}")
print(f"Lift over Baseline: {rf_p50 / 0.240:.2f}x")


=== MODEL BENCHMARK RESULTS ===
Train Set Size: 24,000 rows | Test Set Size: 6,000 rows
Random Forest Precision@50 (Client Holdout): 0.740
Lift over Baseline: 3.08x
